[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-05b-prompt-caching-and-batch-economics.ipynb)

# Advanced Discussion 5 (continued) — prompt caching and batch processing

Cost of a cached prefix, what breaks a cache (measured with a tokenizer), append-only agent history, cache lifetime, and batch pricing. Prices are placeholders.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Pure arithmetic and a tokenizer; runs in seconds.

In [ ]:
%pip install -q tiktoken numpy

In [ ]:
import numpy as np, tiktoken
tok = tiktoken.get_encoding("cl100k_base")
# PLACEHOLDER prices and multipliers: check your provider's current price list. Read multiplier ~0.1x and write multiplier ~1.25x are common shapes, not constants.
IN, OUT = 3.00, 15.00            # dollars per million input / output tokens (placeholder)
READ, WRITE, BATCH = 0.10, 1.25, 0.50

## 1. what caching saves: a support agent with a long fixed prefix and a short changing suffix

In [ ]:
prefix, suffix, out = 6000, 400, 300                   # tokens: system prompt + tools + policy docs ; the user's message ; the answer
def cost(inp_full, inp_cached, out_tokens, write=0):
    return (inp_full * IN + inp_cached * IN * READ + write * IN * WRITE + out_tokens * OUT) / 1e6
nocache = cost(prefix + suffix, 0, out)
first = cost(suffix, 0, out, write=prefix); repeat = cost(suffix, prefix, out)
print("single request, no caching: $%.5f" % nocache)
print("with caching: first request (writes the prefix) $%.5f | later requests (read it) $%.5f (%.0f%% of the no-cache cost)" % (first, repeat, 100 * repeat / nocache))
print("caching pays back from the 2nd request within the lifetime (first request costs %.0f%% more, later ones %.0f%% less)" % (100 * (first / nocache - 1), 100 * (1 - repeat / nocache)))
for n in (1, 2, 3, 5, 10, 100, 1000):
    total_cache = first + (n - 1) * repeat; total_no = n * nocache
    print("  %4d requests within the cache lifetime: no cache $%.4f | cache $%.4f | saving %+.0f%%" % (n, total_no, total_cache, 100 * (1 - total_cache / total_no)))

## 2. what breaks a cache: it matches an exact PREFIX of the token sequence

In [ ]:
sys_prompt = "You are a support agent for Kestrel Logistics. Follow the policy below. " + "Refund requests above 500 need approval. " * 60
tools = " Tools: search_orders(order_id), issue_refund(order_id, amount), escalate(reason). "
def shared_prefix_tokens(a, b):
    ta, tb = tok.encode(a), tok.encode(b); n = 0
    for x, y in zip(ta, tb):
        if x != y: break
        n += 1
    return n, len(ta)
turn1_user, turn2_user = "Where is order 4471?", "Please refund order 4471."
A1 = sys_prompt + tools + "\nCurrent time: 2026-09-30 09:14:07\n" + turn1_user
A2 = sys_prompt + tools + "\nCurrent time: 2026-09-30 09:14:52\n" + turn2_user
B1 = "Current time: 2026-09-30 09:14:07\n" + sys_prompt + tools + "\n" + turn1_user
B2 = "Current time: 2026-09-30 09:14:52\n" + sys_prompt + tools + "\n" + turn2_user
C1 = sys_prompt + tools + "\n" + turn1_user + "\n[time 09:14:07]"
C2 = sys_prompt + tools + "\n" + turn2_user + "\n[time 09:14:52]"
for name, (x, y) in (("timestamp at the very top", (B1, B2)), ("timestamp after the system prompt", (A1, A2)), ("time appended to the user turn", (C1, C2))):
    n, tot = shared_prefix_tokens(x, y); print("  %-36s cacheable shared prefix: %5d of %5d tokens (%.0f%%)" % (name, n, tot, 100 * n / tot))
n, tot = shared_prefix_tokens(sys_prompt + tools + "\nTools list v1", sys_prompt.replace("500", "600", 1) + tools + "\nTools list v1"); print("  editing one number near the start of the policy: shared prefix %d of %d tokens" % (n, tot))

## 3. an agent loop: every turn re-sends the whole history. append-only history caches; rewriting history does not

In [ ]:
turns, tool_result = 12, 700
def agent_cost(cached, rewrite_each_turn=False):
    total = 0.0; hist = prefix
    for t in range(turns):
        new = tool_result + 150
        if not cached: total += cost(hist + new, 0, 120)
        elif rewrite_each_turn: total += cost(hist + new, 0, 120)                        # history edited: prefix no longer matches
        else: total += cost(0, hist, 120, write=new) if t else cost(0, 0, 120, write=hist + new)          # new tokens are written to the cache, the old history is read
        hist += new + 120
    return total, hist
for name, (c, rw) in (("no caching", (False, False)), ("caching, append-only history", (True, False)), ("caching, but history rewritten every turn", (True, True))):
    t, h = agent_cost(c, rw); print("  12-turn agent run, %-42s $%.4f (final context %d tokens)" % (name, t, h))

## 4. cache lifetime: requests arrive as a Poisson stream; the cache expires after a TTL without a hit (a hit refreshes it)

In [ ]:
def hit_rate(rate_per_min, ttl_min, n=20000, seed=0):
    r = np.random.RandomState(seed); gaps = r.exponential(1 / rate_per_min, n); return float(np.mean(gaps <= ttl_min))
print("\nshare of requests that find a warm cache (5-minute lifetime, refreshed by each hit):")
for rate in (0.05, 0.2, 1, 5, 30):
    print("  %5.2f requests/min: %.0f%%" % (rate, 100 * hit_rate(rate, 5)))

## 5. batch processing: half price for patience

In [ ]:
n_req = 10000; per = cost(prefix + suffix, 0, out)
print("\n10,000 non-urgent requests (e.g. nightly classification): real-time $%.0f | batch (50%% discount, results within hours) $%.0f | batch + cached prefix $%.0f" % (
    n_req * per, n_req * per * BATCH, n_req * cost(suffix, prefix, out) * BATCH))
print("what batch costs you: latency (minutes to a day), no streaming, and results may arrive out of order (match on a custom id)")

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.